# 07 · CatBoost categoricals and automatic feature combinations

**Idea:** CatBoost can encode categoricals with ordered target statistics and **combine categorical features** into new ones (e.g. state × package × drivers) as it grows trees. The biggest win so far, `DRIVERS_PER_VEH`, is exactly such an interaction. Two things currently stop CatBoost from doing this:
1. Small-integer columns (drivers, vehicles, CAT zone, year) are numeric, so they never enter combinations.
2. The tuned `one_hot_max_size = 10` one-hot encodes every categorical with ≤ 10 levels (state, carrier, package). One-hot features get **no target statistics and no combinations**.

**New categoricals** (`features.add_cats`, target-free):
- `C_NUM_DRIVERS`, `C_VEH_N_ROWS`, `C_CAT_ZONE`, `C_QUOTE_YEAR`: string copies (numeric originals stay);
- `C_DRV_VEH`: the drivers × vehicles cell, e.g. `3_2`;
- household profiles: each driver's living status / marital status / age band / gender, sorted and joined (`dependent|own|own`).

**Protocol:** tuned CatBoost parameters from `04_tuning`; variants scored on the seed-0 folds (paired Δ vs. the same folds); the best one is confirmed on all 15 folds, judged on the unseen seeds 1–2.

In [1]:
import json, sys
import numpy as np, pandas as pd
sys.path.insert(0, '.')
import common as C, features as FE

lab, hold = C.load()
L = FE.add_cats(FE.add_static(lab))
T = json.load(open('params_tuned.json'))['cb']
F, P = T['features'], T['params']
COPIES = [f'C_{c}' for c in FE.CAT_COPIES]
base = C.cv_score('probe_base', L, F, 'cb', P, seeds=(0,), log=False)   # cached from 06
fmt = {'auc': '{:.4f}', 'Δ auc': '{:+.4f}', 'se Δ': '{:.4f}', 'top_decile': '{:.1%}'}

def run(name, extra=(), **kw):
    r = C.cv_score(f'cats_{name}', L, F + list(extra), 'cb', P | kw, seeds=(0,), log=False)
    d = r['fold_aucs'] - base['fold_aucs']
    return {'auc': r['auc_mean'], 'Δ auc': d.mean(), 'se Δ': d.std() / np.sqrt(5), 'top_decile': r['top_decile_prec'],
            'rounds': r['rounds'], 'n_extra': len(extra)}

## 1. Variants (seed-0 folds)
`ohm` = `one_hot_max_size`; `ohm=1` means no one-hot at all (every categorical gets target statistics and can be combined). `mcc` = `max_ctr_complexity`, the maximum number of categoricals combined into one (CatBoost's CPU default is 4; `mcc=1` disables combinations).

In [2]:
V = {
    'tuned (ohm 10)':                      run('base_ohm10'),
    'ohm 1':                               run('base_ohm1', one_hot_max_size=1),
    '+ copies, ohm 10':                    run('copies_ohm10', COPIES),
    '+ copies, ohm 1':                     run('copies_ohm1', COPIES, one_hot_max_size=1),
    '+ drivers_vehicles cell, ohm 1':      run('drvveh_ohm1', ['C_DRV_VEH'], one_hot_max_size=1),
    '+ profiles, ohm 1':                   run('prof_ohm1', FE.PROFILES, one_hot_max_size=1),
    '+ all, ohm 1':                        run('all_ohm1', COPIES + ['C_DRV_VEH'] + FE.PROFILES, one_hot_max_size=1),
    '+ all, ohm 1, mcc 1 (no combos)':     run('all_ohm1_mcc1', COPIES + ['C_DRV_VEH'] + FE.PROFILES, one_hot_max_size=1, max_ctr_complexity=1),
    '+ all, ohm 1, mcc 2':                 run('all_ohm1_mcc2', COPIES + ['C_DRV_VEH'] + FE.PROFILES, one_hot_max_size=1, max_ctr_complexity=2),
}
for p in FE.PROFILES:
    V[f'+ {p}, ohm 1'] = run(f'{p}_ohm1', [p], one_hot_max_size=1)
tbl = pd.DataFrame(V).T
tbl.style.format(fmt)

,auc,Δ auc,se Δ,top_decile,rounds,n_extra
tuned (ohm 10),0.7130,+0.0000,0.0000,30.0%,485.000000,0.000000
ohm 1,0.7101,-0.0029,0.0012,30.1%,613.000000,0.000000
"+ copies, ohm 10",0.7145,+0.0015,0.0005,30.2%,489.000000,4.000000
"+ copies, ohm 1",0.7141,+0.0011,0.0003,30.3%,1064.000000,4.000000
"+ drivers_vehicles cell, ohm 1",0.7103,-0.0027,0.0007,29.9%,751.000000,1.000000
"+ profiles, ohm 1",0.7108,-0.0022,0.0010,29.8%,679.000000,4.000000
"+ all, ohm 1",0.7132,+0.0002,0.0007,29.9%,869.000000,9.000000
"+ all, ohm 1, mcc 1 (no combos)",0.7117,-0.0013,0.0005,29.6%,801.000000,9.000000
"+ all, ohm 1, mcc 2",0.7133,+0.0003,0.0009,29.9%,600.000000,9.000000
"+ C_LIVING_PROFILE, ohm 1",0.7110,-0.0020,0.0005,30.1%,696.000000,1.000000


**Reading:**
- **Categorical copies of drivers / vehicles / CAT zone / year help (+0.0015 ± 0.0005 with `ohm 10`).** With ≤ 8 levels each, `ohm 10` one-hot encodes them, so the gain is **not** from combinations: CatBoost simply gets a per-level split for each value, which fits the non-monotone, step-shaped effects better (e.g. conversion by drivers × vehicles in `06`) than numeric thresholds alone.
- **Turning one-hot off (`ohm 1`) hurts** the base model (−0.0029): target statistics on state/carrier/package are noisier than one-hot splits.
- **Combinations add nothing:** `+ all` with combos (mcc 2/4) ≈ base; disabling combos (mcc 1) is slightly lower, but neither beats the copies alone.
- **Household profiles and the drivers×vehicles cell hurt** (−0.001 to −0.003): high-cardinality strings get noisy target statistics.

## 2. Confirm the best variant on all 15 folds
The best seed-0 variant (by Δ) is rerun on all 15 folds against the tuned 41-feature model; the decision uses seeds 1–2 only (the folds the selection never saw).

In [3]:
SPEC = {
    'tuned (ohm 10)': ((), {}), 'ohm 1': ((), {'one_hot_max_size': 1}),
    '+ copies, ohm 10': (COPIES, {}), '+ copies, ohm 1': (COPIES, {'one_hot_max_size': 1}),
    '+ drivers_vehicles cell, ohm 1': (['C_DRV_VEH'], {'one_hot_max_size': 1}),
    '+ profiles, ohm 1': (FE.PROFILES, {'one_hot_max_size': 1}),
    '+ all, ohm 1': (COPIES + ['C_DRV_VEH'] + FE.PROFILES, {'one_hot_max_size': 1}),
    '+ all, ohm 1, mcc 1 (no combos)': (COPIES + ['C_DRV_VEH'] + FE.PROFILES, {'one_hot_max_size': 1, 'max_ctr_complexity': 1}),
    '+ all, ohm 1, mcc 2': (COPIES + ['C_DRV_VEH'] + FE.PROFILES, {'one_hot_max_size': 1, 'max_ctr_complexity': 2}),
} | {f'+ {p}, ohm 1': ([p], {'one_hot_max_size': 1}) for p in FE.PROFILES}
ranked = tbl.drop('tuned (ohm 10)').sort_values('Δ auc', ascending=False)
top = ranked.index[:2].tolist()                                     # confirm the two best
base15 = C.cv_score('fe2_base15', L, F, 'cb', P)                     # cached from 06
conf = {}
for name in top:
    extra, kw = SPEC[name]
    r = C.cv_score('cats15_' + ''.join(ch if ch.isalnum() else '_' for ch in name),
                   L, F + list(extra), 'cb', P | kw, notes='07_catboost_cats')
    d = r['fold_aucs'] - base15['fold_aucs']
    conf[name] = {'auc (15)': r['auc_mean'], 'Δ (15)': d.mean(), 'se (15)': d.std() / np.sqrt(15), 'Δ seeds 1-2': d[5:].mean(),
                  'se seeds 1-2': d[5:].std() / np.sqrt(10), 'top_decile': r['top_decile_prec'], 'rounds': r['rounds'], '_r': r}
conf = pd.DataFrame(conf).T
print(f'tuned 41-feature CatBoost, 15 folds: {base15["auc_mean"]:.4f}')
conf.drop(columns='_r').style.format({'auc (15)': '{:.4f}', 'Δ (15)': '{:+.4f}', 'se (15)': '{:.4f}', 'Δ seeds 1-2': '{:+.4f}',
                                      'se seeds 1-2': '{:.4f}', 'top_decile': '{:.1%}'})

tuned 41-feature CatBoost, 15 folds: 0.7118


,auc (15),Δ (15),se (15),Δ seeds 1-2,se seeds 1-2,top_decile,rounds
"+ copies, ohm 10",0.7143,+0.0025,0.0004,+0.0031,0.0004,30.2%,402
"+ copies, ohm 1",0.7133,+0.0016,0.0004,+0.0021,0.0005,30.1%,537


In [4]:
# Adopt only with > 2 se on seeds 1-2; then a 5-seed holdout submission.
c = conf.astype({'Δ seeds 1-2': float, 'se seeds 1-2': float})
win = c[c['Δ seeds 1-2'] > 2 * c['se seeds 1-2']].sort_values('Δ seeds 1-2', ascending=False)
if len(win):
    name = win.index[0]; extra, kw = SPEC[name]
    feats, params, rounds = F + list(extra), P | kw, int(win.loc[name, 'rounds'])
    H = FE.add_cats(FE.add_static(hold))
    preds = np.mean([C.fit_full(L, feats, 'cb', params | {'random_seed': s}, int(round(rounds * 1.2)))(H) for s in [42, 1, 2, 3, 4]], axis=0)
    out = C.write_submission('catboost_cats5', hold, preds)
    json.dump({'variant': name, 'features': feats, 'params': params, 'rounds': rounds, 'cv_auc': float(win.loc[name, 'auc (15)'])},
              open('params_cats.json', 'w'), indent=1, default=str)
    print('adopted', name, '->', out.name, f'| holdout mean pred {preds.mean():.4f}')
else:
    print('no variant passes; tuned 41-feature CatBoost stays')

adopted + copies, ohm 10 -> catboost_cats5_20261030.csv | holdout mean pred 0.1113


## Conclusions
- **Adopted: tuned CatBoost + 4 categorical copies (`C_NUM_DRIVERS`, `C_VEH_N_ROWS`, `C_CAT_ZONE`, `C_QUOTE_YEAR`), `one_hot_max_size = 10`.**
  15-fold CV AUC **0.7143** (from 0.7118): **+0.0031 ± 0.0004 on the unseen seeds 1–2** (~8 se), +0.0025 on all 15; top-decile precision 30.2%; 402 trees.
- Written: `submissions/catboost_cats5_<date>.csv` (5 seeds, trees × 1.2) and `params_cats.json`.
- Mechanism: one-hot per-level splits on small-integer columns, not CatBoost's categorical combinations or target statistics.

**Next:** more one-hot copies of small-integer/ordinal columns; drop the numeric originals; retune CatBoost on the new set (its parameters were tuned without these columns).